<div style="background:#1A1D1D; padding:26px 30px; border-radius:8px;">
<div style="display:inline-block; background:#F05E22; color:#fff; font-family:Calibri,sans-serif;
     font-size:11px; font-weight:bold; letter-spacing:1px; padding:5px 12px; border-radius:12px;">
DAY 5 OF 120</div>
<h1 style="color:#fff; font-family:Cambria,Georgia,serif; margin:14px 0 2px 0; font-size:34px;">
Software Engineering <span style="color:#F05E22;">Habits</span></h1>
<p style="color:#9D9E9E; font-family:Calibri,sans-serif; font-size:14px; margin:8px 0 0 0;">
Testable code, pytest, docstrings, type hints, formatters, and pre-commit hooks.</p>
<p style="color:#F05E22; font-family:Calibri,sans-serif; font-size:12px; margin:14px 0 0 0;">
Phase 1 · Week 1 · Day 5 &nbsp;·&nbsp; CodeTrade AI/ML Internship</p>
</div>

### Why today matters more than it looks

Days 1–4 taught you to write Python and version it. Today teaches you to write Python that other people
— including future-you — can trust, read, and change without fear.

> **This is the difference between code that works once and code that works in production.** In an ML
> job, nobody merges your model if it has no tests, no docstrings, and fails the formatter. These
> habits are not optional polish. They are the entry ticket.

**Today is also the Week 1 deliverable.** By end of day you refactor a messy notebook into a clean,
documented, tested module. The task sheet drives that; this notebook gives you the tools.

This notebook uses `!command` cells — the `!` runs a shell command from inside Jupyter, so you can see
these tools actually run. Run every cell.

---
## 1 · Testable code starts with small functions

Before you can test code, it has to be *testable*. A 200-line function that reads a file, cleans data,
trains a model, and prints results cannot be tested — there is no single thing to check.

> **Testable code** — code split into small functions that each do one thing and return a value.
> If a function returns something, you can assert what it should return.

In [1]:
# HARD to test -- does everything, returns nothing, prints instead
def process_scores_bad(filename):
    with open(filename) as f:
        lines = f.readlines()
    total = 0
    for line in lines:
        total += int(line.strip())
    print("Average:", total / len(lines))   # prints -- can't assert on a print


# EASY to test -- split into pieces that each return a value
def read_scores(filename):
    with open(filename) as f:
        return [int(line.strip()) for line in f]

def average(scores):
    if not scores:
        return 0
    return sum(scores) / len(scores)

# now each piece can be checked independently
print(average([80, 90, 100]))   # 90.0  -- easy to verify
print(average([]))              # 0     -- and the edge case too

90.0
0


The second version is longer. It is also testable, reusable, and debuggable. **Small functions that
return values are the foundation of everything else today.**

---
## 2 · The simplest test: assert

Before frameworks, understand the core idea. A test is just: *"I expect X. Did I get X?"*

> **assert** — a statement that does nothing if the condition is `True`, and raises `AssertionError`
> if it is `False`. That is the whole mechanism behind every test framework.

In [2]:
def average(scores):
    if not scores:
        return 0
    return sum(scores) / len(scores)

# these pass silently -- that is what you want
assert average([80, 90, 100]) == 90.0
assert average([50]) == 50.0
assert average([]) == 0

print("all assertions passed")

all assertions passed


In [3]:
# what a FAILING assertion looks like -- run this to see the error
def buggy_average(scores):
    return sum(scores) / len(scores) + 1   # deliberately wrong

assert buggy_average([80, 90, 100]) == 90.0, "average is off -- check the +1"

AssertionError: average is off -- check the +1

Run that cell. You get `AssertionError: average is off -- check the +1`. The message after the comma is
what makes a failure useful — it tells you *what* was expected without you re-reading the code.

Writing raw `assert` statements works, but it stops at the first failure and has no reporting. That is
what `pytest` fixes.

---
## 3 · pytest: tests that scale

> **pytest** — the standard Python testing framework. You write functions named `test_*`, each
> containing assertions, and pytest finds them, runs them all, and reports which passed and failed.

pytest does not run inside notebook cells the way normal code does — tests live in `.py` files. But we
can write a test file from the notebook and run pytest on it with `!`, so you see the real thing.

In [ ]:
# write a small module to test
module_code = """
def average(scores):
    if not scores:
        return 0
    return sum(scores) / len(scores)

def highest(scores):
    if not scores:
        return None
    return max(scores)
"""

with open("scores.py", "w") as f:
    f.write(module_code)

print("wrote scores.py")

wrote scores.py


In [ ]:
# write the test file -- note the naming: test_*.py, and test_* functions
test_lines = [
    "from scores import average, highest",
    "",
    "def test_average_basic():",
    "    assert average([80, 90, 100]) == 90.0",
    "",
    "def test_average_single():",
    "    assert average([50]) == 50.0",
    "",
    "def test_average_empty():",
    "    assert average([]) == 0          # our chosen behaviour for empty",
    "",
    "def test_highest_basic():",
    "    assert highest([3, 9, 2]) == 9",
    "",
    "def test_highest_empty():",
    "    assert highest([]) is None",
]

with open("test_scores.py", "w") as f:
    f.write("\n".join(test_lines))

print("wrote test_scores.py")

wrote test_scores.py


In [ ]:
# run pytest and see the report
!{sys.executable} -m pip install pytest -q --break-system-packages
!{sys.executable} -m pytest test_scores.py -v

============================= test session starts =============================
platform win32 -- Python 3.11.15, pytest-9.1.1, pluggy-1.6.0 -- C:\Users\vikra\Anaconda3\envs\rag_agent\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\vikra\Downloads
plugins: anyio-4.14.1
collecting ... collected 5 items

test_scores.py::test_average_basic PASSED                                [ 20%]
test_scores.py::test_average_single PASSED                               [ 40%]
test_scores.py::test_average_empty PASSED                                [ 60%]
test_scores.py::test_highest_basic PASSED                                [ 80%]
test_scores.py::test_highest_empty PASSED                                [100%]

============================== 5 passed in 0.07s ==============================


Read that output. Five tests, five green `PASSED`. Each test name describes exactly what it checks —
`test_average_empty` tells you the edge case is covered without opening the file.

**Test the edges, not just the happy path.** Empty lists, single elements, zeros, negatives, wrong
types — that is where bugs live. `test_average_basic` is the easy one; `test_average_empty` is the one
that saves you.

In [ ]:
# add a failing test to see how pytest reports failures
failing_lines = [
    "from scores import average",
    "",
    "def test_this_will_fail():",
    "    assert average([10, 20]) == 99   # wrong on purpose",
]
with open("test_fail_demo.py", "w") as f:
    f.write("\n".join(failing_lines))

!{sys.executable} -m pytest test_fail_demo.py -v

============================= test session starts =============================
platform win32 -- Python 3.11.15, pytest-9.1.1, pluggy-1.6.0 -- C:\Users\vikra\Anaconda3\envs\rag_agent\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\vikra\Downloads
plugins: anyio-4.14.1
collecting ... collected 1 item

test_fail_demo.py::test_this_will_fail FAILED                            [100%]

================================== FAILURES ===================================
_____________________________ test_this_will_fail _____________________________

    def test_this_will_fail():
>       assert average([10, 20]) == 99   # wrong on purpose
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
E       assert 15.0 == 99
E        +  where 15.0 = average([10, 20])

test_fail_demo.py:4: AssertionError
=========================== short test summary info ===========================
FAILED test_fail_demo.py::test_this_will_fail - assert 15.0 == 99
============================== 1 failed in 0.79s ===================

When a test fails, pytest shows you the exact line, what it expected (`99`), and what it got (`15.0`).
No guessing. This is why `assert average(...) == 99` beats a `print` — the framework does the comparing
and the reporting for you.

---
## 4 · Docstrings: what and why, not how

> **Docstring** — a string literal right under a `def` or `class` that documents it. Tools, editors,
> and `help()` read it automatically. The code says *how*; the docstring says *what* and *why*.

In [ ]:
def average(scores):
    """Return the arithmetic mean of a list of numbers.

    Args:
        scores: A list of numbers. May be empty.

    Returns:
        The mean as a float, or 0 if the list is empty.
    """
    if not scores:
        return 0
    return sum(scores) / len(scores)


# the docstring is not a comment -- it is accessible at runtime
print(average.__doc__)
print("-" * 40)
help(average)

Return the arithmetic mean of a list of numbers.

    Args:
        scores: A list of numbers. May be empty.

    Returns:
        The mean as a float, or 0 if the list is empty.
    
----------------------------------------
Help on function average in module __main__:

average(scores)
    Return the arithmetic mean of a list of numbers.
    
    Args:
        scores: A list of numbers. May be empty.
    
    Returns:
        The mean as a float, or 0 if the list is empty.



That is the **Google style** docstring — Args, Returns, and a one-line summary first. There is also
NumPy style (used heavily in data science). Pick one and be consistent.

**What makes a docstring good:**
- The first line is a short summary, in the imperative: *"Return the mean"*, not *"This function returns the mean"*
- Document the *arguments* and the *return value*
- Note edge cases and anything surprising (here: empty list returns 0, not an error)
- Do **not** narrate the obvious. `# add 1 to x` above `x += 1` is noise.

In [ ]:
# comment vs docstring -- they are different tools
def clean_name(raw):
    """Strip whitespace and title-case a name.

    Args:
        raw: A possibly-messy name string, e.g. "  aSHa  ".

    Returns:
        A cleaned name, e.g. "Asha".
    """
    # .strip() first so title-casing doesn't choke on leading spaces
    # (this comment explains a non-obvious ORDERING decision -- worth it)
    return raw.strip().title()

print(clean_name("  aSHa  "))

Asha


Notice the split: the **docstring** says what the function does for anyone calling it. The **comment**
explains a non-obvious internal decision for anyone editing it. Both earn their place. A comment that
just restates the code does not.

---
## 5 · Type hints: documentation the tools can check

> **Type hint** — an annotation of what type a variable, argument, or return value is expected to be.
> Python does not enforce them at runtime, but they document intent and let tools like `mypy` catch
> mistakes before you run the code.

In [ ]:
# without hints -- what goes in? what comes out? you have to read the body
def average_untyped(scores):
    return sum(scores) / len(scores)

# with hints -- the signature tells the whole story
def average(scores: list[float]) -> float:
    """Return the mean of a list of numbers."""
    if not scores:
        return 0.0
    return sum(scores) / len(scores)

print(average([1.0, 2.0, 3.0]))

# hints are readable at runtime too
print(average.__annotations__)

2.0
{'scores': list[float], 'return': <class 'float'>}


Hints do not stop you passing the wrong type — Python will still run `average("hello")` and crash
inside. What they give you is:

- A **signature that documents itself** — `list[float] -> float` says everything
- **Editor autocomplete** that actually knows what your variables are
- **mypy** catching type errors statically, before the code ever runs

Common hints you will use constantly: `int`, `str`, `float`, `bool`, `list[int]`, `dict[str, int]`,
`tuple[int, int]`, and `Optional[X]` (meaning "X or None").

In [ ]:
from typing import Optional

def find_student(roll: int, students: dict[int, str]) -> Optional[str]:
    """Return the student name for a roll number, or None if not found."""
    return students.get(roll)

roster = {1: "Asha", 2: "Ravi"}
print(find_student(1, roster))    # Asha
print(find_student(99, roster))   # None -- hence Optional[str]

Asha
None


---
## 6 · Formatters: stop arguing about style

> **black** — an opinionated code formatter. It rewrites your code into one canonical style. No config,
> no debates: everyone's code looks the same, so diffs show real changes, not spacing.

> **ruff** — an extremely fast linter (and formatter). It flags unused imports, undefined names, bad
> patterns, and style issues — the things that cause bugs and noise.

Let's watch black reformat deliberately-ugly code.

In [ ]:
ugly_lines = [
    "def   messy(x,y):",
    "    z=[1,2,3,4,5]",
    "    result=sum( z )+x+y",
    "    return result",
    "d={'a':1,'b':2}",
]

with open("ugly.py", "w") as f:
    f.write("\n".join(ugly_lines) + "\n")

print("BEFORE black:")
print(open("ugly.py").read())

BEFORE black:
def   messy(x,y):
    z=[1,2,3,4,5]
    result=sum( z )+x+y
    return result
d={'a':1,'b':2}



In [ ]:
!{sys.executable} -m pip install black -q --break-system-packages
!{sys.executable} -m black ugly.py
print("\nAFTER black:")
print(open("ugly.py").read())


AFTER black:
def messy(x, y):
    z = [1, 2, 3, 4, 5]
    result = sum(z) + x + y
    return result


d = {"a": 1, "b": 2}



reformatted ugly.py

All done! \u2728 \U0001f370 \u2728
1 file reformatted.


Look at what black fixed automatically: spacing around `=`, `,`, and operators; consistent quotes;
proper blank lines. You did nothing but run one command. Multiply that across a whole team and every
file in the repo looks identical — so code review focuses on *logic*, never on whitespace.

In [ ]:
# ruff catches problems black doesn't -- unused imports, undefined names
problem_lines = [
    "import os",
    "import sys",
    "import json",
    "",
    "def greet(name):",
    '    return f"Hello {nam}"     # typo: \'nam\' is undefined',
]
with open("problematic.py", "w") as f:
    f.write("\n".join(problem_lines) + "\n")

!{sys.executable} -m pip install ruff -q --break-system-packages
!{sys.executable} -m ruff check problematic.py

F401 [*] `os` imported but unused
 --> problematic.py:1:8
  |
1 | import os
  |        ^^
2 | import sys
3 | import json
  |
help: Remove unused import: `os`

F401 [*] `sys` imported but unused
 --> problematic.py:2:8
  |
1 | import os
2 | import sys
  |        ^^^
3 | import json
  |
help: Remove unused import: `sys`

F401 [*] `json` imported but unused
 --> problematic.py:3:8
  |
1 | import os
2 | import sys
3 | import json
  |        ^^^^
4 |
5 | def greet(name):
  |
help: Remove unused import: `json`

F821 Undefined name `nam`
 --> problematic.py:6:21
  |
5 | def greet(name):
6 |     return f"Hello {nam}"     # typo: 'nam' is undefined
  |                     ^^^
  |

Found 4 errors.
[*] 3 fixable with the `--fix` option.


ruff caught two classes of problem: three **unused imports** (`os`, `sys`, `json` — imported, never
used) and an **undefined name** (`nam`, a typo for `name`). That last one is a real bug that would have
crashed at runtime. ruff found it in milliseconds without running the code.

---
## 7 · pre-commit hooks: make good habits automatic

Knowing about black, ruff, and pytest does nothing if you forget to run them. The fix is to make them
run **automatically, every time you commit.**

> **pre-commit hook** — a script Git runs automatically before it lets a commit go through. If the
> hook fails (a test breaks, the formatter finds unformatted code), the commit is blocked until you fix
> it. Bad code never even enters your history.

Hooks are configured in a file called `.pre-commit-config.yaml`. You cannot fully run this from a
notebook (it needs a Git repo), so this is a **preview** — you set it up for real in the task sheet.

In [ ]:
# this is what a .pre-commit-config.yaml looks like
config = """
repos:
  - repo: https://github.com/psf/black
    rev: 24.10.0
    hooks:
      - id: black

  - repo: https://github.com/astral-sh/ruff-pre-commit
    rev: v0.8.0
    hooks:
      - id: ruff
"""
print(config)
print("You'll create this file and run 'pre-commit install' in the task sheet.")


repos:
  - repo: https://github.com/psf/black
    rev: 24.10.0
    hooks:
      - id: black

  - repo: https://github.com/astral-sh/ruff-pre-commit
    rev: v0.8.0
    hooks:
      - id: ruff

You'll create this file and run 'pre-commit install' in the task sheet.


Once installed, the flow becomes: you type `git commit`, and *before* the commit is saved, black
reformats your code and ruff checks it. If ruff finds that undefined-name bug from earlier, **the commit
is rejected** and you fix it first. The bad code never reaches GitHub.

This is the whole philosophy of today: **don't rely on remembering to be careful. Build a system that
is careful for you.**

---
## 8 · All of it: a clean module

Here is what "production-quality" looks like for a small module — small typed functions, docstrings,
and a matching test file. This is the shape of your Week 1 deliverable.

In [ ]:
clean_lines = [
    '"""Utilities for working with student scores."""',
    "",
    "",
    "def read_scores(path: str) -> list[int]:",
    '    """Read integer scores from a file, one per line.',
    "",
    "    Args:",
    "        path: Path to a text file with one integer per line.",
    "",
    "    Returns:",
    "        A list of integer scores.",
    '    """',
    "    with open(path) as f:",
    "        return [int(line.strip()) for line in f if line.strip()]",
    "",
    "",
    "def average(scores: list[int]) -> float:",
    '    """Return the mean of the scores, or 0.0 if the list is empty."""',
    "    if not scores:",
    "        return 0.0",
    "    return sum(scores) / len(scores)",
    "",
    "",
    "def passed(scores: list[int], cutoff: int = 40) -> list[int]:",
    '    """Return only the scores at or above the cutoff.',
    "",
    "    Args:",
    "        scores: The scores to filter.",
    "        cutoff: The minimum passing score, inclusive. Defaults to 40.",
    "",
    "    Returns:",
    "        The subset of scores that passed.",
    '    """',
    "    return [s for s in scores if s >= cutoff]",
]

with open("student_utils.py", "w") as f:
    f.write("\n".join(clean_lines))
print("wrote student_utils.py")

wrote student_utils.py


In [ ]:
clean_test_lines = [
    "from student_utils import average, passed",
    "",
    "def test_average_basic():",
    "    assert average([80, 90, 100]) == 90.0",
    "",
    "def test_average_empty():",
    "    assert average([]) == 0.0",
    "",
    "def test_passed_default_cutoff():",
    "    assert passed([30, 40, 50]) == [40, 50]   # 40 is inclusive!",
    "",
    "def test_passed_custom_cutoff():",
    "    assert passed([30, 40, 50], cutoff=45) == [50]",
    "",
    "def test_passed_none_pass():",
    "    assert passed([10, 20, 30]) == []",
]

with open("test_student_utils.py", "w") as f:
    f.write("\n".join(clean_test_lines))

!{sys.executable} -m pytest test_student_utils.py -v

============================= test session starts =============================
platform win32 -- Python 3.11.15, pytest-9.1.1, pluggy-1.6.0 -- C:\Users\vikra\Anaconda3\envs\rag_agent\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\vikra\Downloads
plugins: anyio-4.14.1
collecting ... collected 5 items

test_student_utils.py::test_average_basic PASSED                         [ 20%]
test_student_utils.py::test_average_empty PASSED                         [ 40%]
test_student_utils.py::test_passed_default_cutoff PASSED                 [ 60%]
test_student_utils.py::test_passed_custom_cutoff PASSED                  [ 80%]
test_student_utils.py::test_passed_none_pass PASSED                      [100%]

============================== 5 passed in 0.08s ==============================


Notice `test_passed_default_cutoff` checks that 40 **passes** with a cutoff of 40 — the inclusive
boundary. That is exactly the off-by-one you hunted in the Day 2 debugging task. A test locks that
decision in place so nobody accidentally breaks it later.

---
## 9 · Recap

| Habit | Tool | Why |
|---|---|---|
| Small functions that return values | — | Nothing else is testable without this |
| Test the edges | `assert`, `pytest` | Bugs live in empty lists and boundaries |
| Say what and why | Docstrings | Future-you and teammates need the intent |
| Declare your types | Type hints, `mypy` | Self-documenting, catches errors early |
| One canonical style | `black` | Diffs show logic, not whitespace |
| Catch bad patterns | `ruff` | Unused imports, undefined names, bugs |
| Make it automatic | pre-commit hooks | Don't rely on remembering |

### The one idea behind all of it

**Build systems that make the right thing automatic.** You will not remember to run the formatter every
time. You will not remember to test every edge. So you set up tools that do it for you — and then you
cannot forget. That is what separates a professional from someone who merely writes code that runs.

<div style="background:#FDF0E9; border-left:4px solid #F05E22; padding:16px 20px; border-radius:5px;">
<b style="color:#F05E22; font-family:Calibri,sans-serif;">Now open the task sheet — Week 1 deliverable</b>
<p style="font-family:Calibri,sans-serif; font-size:13px; color:#1A1D1D; margin:6px 0 0 0;">
You will take a messy notebook, refactor it into a clean typed module with docstrings, write a real
pytest suite, set up pre-commit hooks, and get it all passing in CI. This is 2% of your program grade
and the capstone of Week 1.</p>
</div>